# 문서 이미지 자동 교정 및 향상 시스템
# Document Image Auto-Correction & Enhancement

**Google Colab Edition**

스마트폰으로 촬영된 문서 이미지의 기하학적 왜곡을 보정하고 텍스트 가독성을 향상시키는 시스템입니다.

- ⚠️ Machine Learning / Deep Learning 사용 금지
- ✅ 전통적 Computer Vision + Image Processing 기법만 사용 (OpenCV, NumPy)

## Pipeline (11 Steps)

```
Input → Grayscale → Gaussian Blur → Canny Edge → Edge Dilation
      → Contour Detection → Convex Hull → Quad Approximation
      → Corner Ordering → Perspective Transform
      → CLAHE → Adaptive Threshold → Morphology → Output
```

In [ ]:
# 1. 환경 설정 및 라이브러리 import
# Google Colab에는 OpenCV, NumPy, Matplotlib가 기본 설치되어 있습니다.
# 만약 누락된 패키지가 있다면 아래 주석을 해제하세요:
# !pip install opencv-python numpy matplotlib

import cv2
import numpy as np
import os
import time
import glob
from pathlib import Path
import matplotlib.pyplot as plt
from google.colab import files
from google.colab.patches import cv2_imshow
from IPython.display import display, Image as IPImage, HTML
import ipywidgets as widgets
from IPython.display import clear_output

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")
print("All libraries loaded successfully!")

---
## 2. Utility Functions (Image I/O)

In [ ]:
# 2. 이미지 입출력 유틸리티

def load_image(path):
    """Load image from file path as BGR ndarray. Uses: cv2.imread()"""
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Image file not found: {path}")
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Failed to read image: {path}")
    return img


def save_image(img, path):
    """Save image to disk. Uses: cv2.imwrite()"""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    cv2.imwrite(path, img)


def save_step(img, output_dir, step_num, step_name):
    """Save intermediate result as '{step_num:02d}_{step_name}.jpg'. Uses: cv2.imwrite()"""
    filename = f"{step_num:02d}_{step_name}.jpg"
    filepath = os.path.join(output_dir, filename)
    os.makedirs(output_dir, exist_ok=True)
    cv2.imwrite(filepath, img)


def show_images(images, titles, cols=3, figsize=(18, 10)):
    """Display multiple images in a grid using matplotlib."""
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten() if rows > 1 or cols > 1 else [axes]
    for i, (img, title) in enumerate(zip(images, titles)):
        if len(img.shape) == 2:
            axes[i].imshow(img, cmap='gray')
        else:
            axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[i].set_title(title, fontsize=10)
        axes[i].axis('off')
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()

print("Utility functions defined.")

---
## 3. Preprocessing Module
Step 2-3: Grayscale Conversion + Gaussian Filtering

In [ ]:
# 3. 전처리 모듈

def to_grayscale(img):
    """Convert BGR to grayscale. Formula: Gray = 0.299R + 0.587G + 0.114B
    Uses: cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)"""
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)


def apply_gaussian_blur(gray, kernel_size=(5, 5), sigma=0):
    """Apply Gaussian filter. Uses: cv2.GaussianBlur(gray, kernel_size, sigma)"""
    return cv2.GaussianBlur(gray, kernel_size, sigma)

print("Preprocessing functions defined.")

---
## 4. Detection Module
Step 4-6: Canny Edge → Edge Dilation → Contour Detection → Quadrilateral Approximation

In [ ]:
# 4. 문서 검출 모듈

def detect_edges(blurred, low=75, high=200):
    """Extract edges. Uses: cv2.Canny(blurred, low, high)"""
    return cv2.Canny(blurred, low, high)


def dilate_edges(edges, kernel_size=(3, 3), iterations=2):
    """Dilate edges to connect fragments. Uses: cv2.dilate()"""
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, kernel_size)
    return cv2.dilate(edges, kernel, iterations=iterations)


def find_contours(edges):
    """Extract contours sorted by area (top 10).
    Uses: cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)"""
    contours, _ = cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    return contours[:10]


def _approx_quad(contour):
    """Try polygon approximation at various epsilon ratios. Uses: cv2.approxPolyDP()"""
    peri = cv2.arcLength(contour, True)
    for ratio in (0.02, 0.03, 0.05, 0.08, 0.10):
        epsilon = ratio * peri
        approx = cv2.approxPolyDP(contour, epsilon, True)
        if len(approx) == 4:
            return approx
    return None


def find_document_contour(contours, img_area=0):
    """Best quadrilateral document candidate.
    Strategy:
      1. Direct polygon approximation per contour.
      2. Convex hull + approximation for partial edges near frame.
      3. Combined contour convex hull as fallback.
    Uses: cv2.approxPolyDP(), cv2.convexHull(), cv2.contourArea()
    """
    min_area = img_area * 0.02 if img_area > 0 else 5000

    for contour in contours:
        area = cv2.contourArea(contour)
        if area < min_area:
            continue
        quad = _approx_quad(contour)
        if quad is not None:
            return quad
        hull = cv2.convexHull(contour)
        quad = _approx_quad(hull)
        if quad is not None:
            return quad

    if len(contours) >= 2:
        combined = np.vstack(contours)
        hull = cv2.convexHull(combined)
        hull_area = cv2.contourArea(hull)
        if hull_area >= min_area:
            quad = _approx_quad(hull)
            if quad is not None:
                return quad
    return None


def draw_contour(img, contour):
    """Draw green boundary. Uses: cv2.drawContours()"""
    result = img.copy()
    cv2.drawContours(result, [contour], -1, (0, 255, 0), 3)
    return result

print("Detection functions defined.")

---
## 5. Geometry Module
Step 7-8: Corner Ordering + Perspective Transformation

In [ ]:
# 5. 기하학적 보정 모듈

def _dist(p1, p2):
    """Euclidean distance."""
    return np.sqrt((p2[0] - p1[0]) ** 2 + (p2[1] - p1[1]) ** 2)


def order_points(pts):
    """Order 4 corners as [TL, TR, BR, BL].
    TL: min x+y, BR: max x+y, TR: max x-y, BL: min x-y.
    Returns (4,2) float32 ndarray."""
    pts = pts.reshape(4, 2)
    ordered = np.zeros((4, 2), dtype=np.float32)
    sums = pts.sum(axis=1)
    diffs = np.diff(pts, axis=1)
    ordered[0] = pts[np.argmin(sums)]   # TL
    ordered[2] = pts[np.argmax(sums)]   # BR
    ordered[1] = pts[np.argmax(diffs)]  # TR
    ordered[3] = pts[np.argmin(diffs)]  # BL
    return ordered


def calculate_output_size(ordered_pts):
    """Compute output width and height.
    width  = max(dist(BL,BR), dist(TL,TR))
    height = max(dist(TR,BR), dist(TL,BL))"""
    tl, tr, br, bl = ordered_pts
    width = int(max(_dist(bl, br), _dist(tl, tr)))
    height = int(max(_dist(tr, br), _dist(tl, bl)))
    return width, height


def apply_perspective_transform(img, ordered_pts):
    """Straighten document via perspective transform.
    Uses: cv2.getPerspectiveTransform(), cv2.warpPerspective()"""
    width, height = calculate_output_size(ordered_pts)
    dst_pts = np.array([
        [0, 0],
        [width - 1, 0],
        [width - 1, height - 1],
        [0, height - 1],
    ], dtype=np.float32)
    M = cv2.getPerspectiveTransform(ordered_pts, dst_pts)
    warped = cv2.warpPerspective(img, M, (width, height))
    return warped

print("Geometry functions defined.")

---
## 6. Enhancement Module
Step 9-10: CLAHE + Adaptive Thresholding + Morphological Processing

In [ ]:
# 6. 이미지 향상 모듈

def apply_clahe(img_gray, clip_limit=2.0, tile_size=(8, 8)):
    """CLAHE local contrast enhancement.
    Uses: cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)"""
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
    return clahe.apply(img_gray)


def apply_adaptive_threshold(img_gray, block_size=11, C=2):
    """Adaptive binarization for uneven illumination.
    Uses: cv2.adaptiveThreshold()"""
    return cv2.adaptiveThreshold(
        img_gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        block_size, C,
    )


def apply_morphology(img_binary, kernel_size=(3, 3)):
    """Morphological closing. Uses: cv2.morphologyEx(MORPH_CLOSE)"""
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, kernel_size)
    return cv2.morphologyEx(img_binary, cv2.MORPH_CLOSE, kernel)


def enhance(warped):
    """Full enhancement: grayscale → CLAHE → adaptive threshold → morphology.
    Returns BGR image."""
    gray = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
    clahed = apply_clahe(gray)
    thresh = apply_adaptive_threshold(clahed)
    cleaned = apply_morphology(thresh)
    return cv2.cvtColor(cleaned, cv2.COLOR_GRAY2BGR)

print("Enhancement functions defined.")

---
## 7. Full Pipeline

In [ ]:
# 7. 전체 파이프라인

def run_pipeline(image, output_dir="output", save_steps=True, basename="document"):
    """Run the full 11-step document correction pipeline.

    Args:
        image: BGR image (numpy array) or path string.
        output_dir: Directory for output images.
        save_steps: Save intermediate step images.
        basename: Base name for output files.

    Returns:
        dict: {success, output_path, processing_time, steps_images, warped, result}
    """
    start_time = time.time()
    output_path = os.path.join(output_dir, f"{basename}_corrected.jpg")
    steps_dir = os.path.join(output_dir, f"{basename}_steps") if save_steps else None
    steps_images = {}

    # Load image if path provided
    if isinstance(image, str):
        original = load_image(image)
    else:
        original = image.copy()

    if save_steps:
        save_step(original, steps_dir, 1, "original")
        steps_images['01_original'] = original.copy()

    # Step 2: Grayscale
    gray = to_grayscale(original)
    if save_steps:
        save_step(gray, steps_dir, 2, "grayscale")
        steps_images['02_grayscale'] = gray.copy()

    # Step 3: Gaussian blur
    blurred = apply_gaussian_blur(gray)
    if save_steps:
        save_step(blurred, steps_dir, 3, "blurred")
        steps_images['03_blurred'] = blurred.copy()

    # Step 4: Canny edge detection
    edges = detect_edges(blurred)
    if save_steps:
        save_step(edges, steps_dir, 4, "edges")
        steps_images['04_edges'] = edges.copy()

    # Step 4b: Dilate edges
    dilated = dilate_edges(edges)

    # Step 5-6: Contour detection + quadrilateral approximation
    contours = find_contours(dilated)
    img_area = original.shape[0] * original.shape[1]
    doc_contour = find_document_contour(contours, img_area)

    if save_steps and contours:
        display_contour = doc_contour if doc_contour is not None else contours[0]
        contour_img = draw_contour(original, display_contour)
        save_step(contour_img, steps_dir, 5, "contours")
        steps_images['05_contours'] = contour_img.copy()

    if doc_contour is not None:
        # Step 7-8: Corner ordering + perspective transform
        ordered_pts = order_points(doc_contour)
        warped = apply_perspective_transform(original, ordered_pts)
        if save_steps:
            save_step(warped, steps_dir, 6, "perspective")
            steps_images['06_perspective'] = warped.copy()

        # Step 9-10: Enhancement
        result = enhance(warped)
        if save_steps:
            save_step(result, steps_dir, 7, "enhanced")
            steps_images['07_enhanced'] = result.copy()

        success = True
    else:
        # Detection failed — fallback to enhancement on original
        result = enhance(original)
        if save_steps:
            save_step(original, steps_dir, 6, "perspective")
            save_step(result, steps_dir, 7, "enhanced")
            steps_images['06_perspective'] = original.copy()
            steps_images['07_enhanced'] = result.copy()
        warped = original
        success = False

    save_image(result, output_path)
    elapsed = time.time() - start_time

    return {
        "success": success,
        "output_path": output_path,
        "processing_time": elapsed,
        "steps_images": steps_images,
        "warped": warped,
        "result": result,
        "original": original,
    }

print("Pipeline function defined.")

---
## 8. 문서 이미지 업로드 및 처리

아래 셀을 실행하여 이미지를 업로드하고 처리하세요.

### 옵션 A: 직접 업로드 (권장)
아래 버튼을 클릭하여 이미지 파일을 선택하세요. 여러 장 선택 가능합니다.

In [ ]:
# 8-A. 이미지 업로드 (JPG, PNG, BMP 지원)

print("이미지 파일을 선택하세요 (여러 장 가능)...")
uploaded = files.upload()

input_images = {}
for fname, data in uploaded.items():
    if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
        # Save uploaded file
        with open(fname, 'wb') as f:
            f.write(data)
        img = cv2.imread(fname)
        if img is not None:
            input_images[fname] = img
            print(f"  [OK] {fname} — {img.shape[1]}x{img.shape[0]}")
        else:
            print(f"  [FAIL] {fname} — could not read as image")
    else:
        print(f"  [SKIP] {fname} — unsupported format")

if not input_images:
    print("\n⚠️ No valid images uploaded. Please upload JPG, PNG, or BMP files.")
else:
    print(f"\n✅ {len(input_images)} image(s) ready for processing.")

### 옵션 B: Google Drive에서 이미지 가져오기
Google Drive를 마운트하고 이미지 경로를 지정하세요.

In [ ]:
# 8-B. (선택) Google Drive 마운트
# 필요한 경우에만 아래 코드를 실행하세요.

use_drive = False  # True로 변경하여 Google Drive 사용

if use_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # 이미지가 있는 폴더 경로 (자신의 경로로 변경하세요)
    DRIVE_IMAGE_DIR = "/content/drive/MyDrive/document_images"
    
    if os.path.isdir(DRIVE_IMAGE_DIR):
        input_images = {}
        for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp'):
            for fpath in glob.glob(os.path.join(DRIVE_IMAGE_DIR, ext)):
                fpath_lower = fpath.lower()
                if fpath_lower.endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                    img = cv2.imread(fpath)
                    if img is not None:
                        fname = os.path.basename(fpath)
                        input_images[fname] = img
                        print(f"  [OK] {fname}")
        print(f"\n✅ {len(input_images)} image(s) loaded from Drive.")
    else:
        print(f"⚠️ Directory not found: {DRIVE_IMAGE_DIR}")
        print("Please update DRIVE_IMAGE_DIR to your actual image folder path.")

---
## 9. 파이프라인 실행

업로드한 모든 이미지를 처리하고 결과를 출력합니다.

In [ ]:
# 9. 업로드된 모든 이미지에 대해 파이프라인 실행

if 'input_images' not in dir() or not input_images:
    print("⚠️ 먼저 이미지를 업로드하세요! (옵션 A 또는 B 실행)")
else:
    OUTPUT_DIR = "output"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    success_count = 0
    total_time = 0.0
    all_results = []
    
    for fname, img in input_images.items():
        basename = os.path.splitext(fname)[0]
        print(f"{'='*60}")
        print(f"[INFO] Processing: {fname} ({img.shape[1]}x{img.shape[0]})")
        
        result = run_pipeline(img, OUTPUT_DIR, save_steps=True, basename=basename)
        
        print(f"[INFO] Document detected: {result['success']}")
        print(f"[INFO] Processing time: {result['processing_time']:.3f}s")
        print(f"[INFO] Saved: {result['output_path']}")
        
        if result['success']:
            success_count += 1
        total_time += result['processing_time']
        all_results.append((basename, result))
    
    total = len(input_images)
    print(f"\n{'='*60}")
    print(f"[SUMMARY] Detection success: {success_count}/{total} ({100*success_count/total:.1f}%)")
    print(f"[SUMMARY] Total processing time: {total_time:.3f}s")
    print(f"[SUMMARY] Average time: {total_time/total:.3f}s/image")

---
## 10. 처리 결과 시각화

각 이미지의 단계별 중간 결과를 확인합니다. 드롭다운으로 이미지를 선택하세요.

In [ ]:
# 10. 결과 시각화

if 'all_results' not in dir() or not all_results:
    print("⚠️ 먼저 파이프라인을 실행하세요!")
else:
    for basename, result in all_results:
        print(f"\n{'='*60}")
        status_emoji = "✅" if result['success'] else "⚠️"
        print(f"{status_emoji} {basename} — {result['processing_time']:.3f}s")
        
        # Original vs Result comparison
        original = result['original']
        final = result['result']
        warped = result.get('warped')
        
        images = [original]
        titles = ["01. Original"]
        
        if warped is not None and result['success']:
            images.append(warped)
            titles.append("06. Perspective Corrected")
        
        images.append(final)
        titles.append("07. Enhanced (Final)")
        
        show_images(images, titles, cols=len(images), figsize=(6*len(images), 5))
        
        # Show all 7 steps if available
        steps = result.get('steps_images', {})
        if steps:
            step_names = ['01_original', '02_grayscale', '03_blurred', 
                         '04_edges', '05_contours', '06_perspective', '07_enhanced']
            step_images = [steps[name] for name in step_names if name in steps]
            step_titles = [name.replace('_', ' ').title() for name in step_names if name in steps]
            if step_images:
                print(f"\n  --- All 7 Steps for {basename} ---")
                show_images(step_images, step_titles, cols=4, figsize=(16, 10))

---
## 11. 결과 파일 다운로드

처리된 이미지와 중간 결과를 ZIP 파일로 다운로드합니다.

In [ ]:
# 11. 결과 다운로드

import shutil

if os.path.isdir(OUTPUT_DIR if 'OUTPUT_DIR' in dir() else 'output'):
    output_zip = 'document_correction_results.zip'
    shutil.make_archive('document_correction_results', 'zip', 
                        OUTPUT_DIR if 'OUTPUT_DIR' in dir() else 'output')
    files.download(output_zip)
    print(f"✅ Results downloaded as: {output_zip}")
else:
    print("⚠️ No output directory found. Run the pipeline first.")

---
## 12. 사용 방법 요약

1. **환경 설정**: 상단의 모든 코드 셀을 순서대로 실행 (`Runtime → Run all`)
2. **이미지 업로드**: 셀 8-A가 실행되면 이미지 파일 선택 (JPG/PNG/BMP)
3. **처리 실행**: 셀 9가 자동으로 모든 이미지 처리
4. **결과 확인**: 셀 10에서 단계별 결과 시각화
5. **다운로드**: 셀 11에서 ZIP 파일 다운로드

### 처리 과정

| Step | 설명 |
|------|------|
| 01 | Original Image (원본) |
| 02 | Grayscale Conversion (그레이스케일) |
| 03 | Gaussian Blur (가우시안 블러) |
| 04 | Canny Edge Detection (에지 검출) |
| 05 | Contour Detection (윤곽선 검출) |
| 06 | Perspective Transform (투시 변환) |
| 07 | Enhanced Result (향상 결과) |

### 파라미터 조정

검출 성능을 개선하려면 아래 파라미터를 조정해보세요:
- Canny thresholds: `detect_edges(blurred, low=50, high=150)`
- Gaussian kernel: `apply_gaussian_blur(gray, kernel_size=(7,7))`
- CLAHE clip limit: `apply_clahe(gray, clip_limit=3.0)`